In [20]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np
import dtale

from scipy.stats import chi2_contingency


In [ ]:
# Rutas
RUTA_LECTURA = 
RUTA_ESCRITURA = 

In [ ]:
# Cargar datos 
df_marketing = pd.read_csv(RUTA_LECTURA, sep=";")

In [ ]:
# Convertir columnas binarias a numéricas (1 para 'yes', 0 para 'no')
columnas_binarias = ['deposit', 'default', 'housing', 'loan']
for col in columnas_binarias:
    df_marketing[col] = df_marketing[col].map({'yes': 1, 'no': 0})


In [37]:
display(df_marketing.dtypes)

id                       int64
age                      int64
job                     object
marital                 object
education               object
default                  int64
balance                  int64
housing                  int64
loan                     int64
contact                 object
day                      int64
month                   object
duration                 int64
campaign                 int64
pdays                    int64
previous                 int64
poutcome                object
deposit                  int64
debt_profile            object
balance_tier            object
job_group               object
month_num                int64
year                     int64
date            datetime64[ns]
day_of_week           category
dtype: object

In [22]:
# --- CREACIÓN DE VARIABLES NECESARIAS (porque estaba usando archivo raw, lo puedes actualizar al cargar el csv clean) ---



# Perfil de deuda 
df_marketing['debt_profile'] = np.where(
    (df_marketing['housing'] == 1) | (df_marketing['loan'] == 1), 'With Debt', 'No Debt')

# Tramos de Balance 
condiciones_balance = [
    (df_marketing['balance'] <= 0),
    (df_marketing['balance'] > 0) & (df_marketing['balance'] <= 556),
    (df_marketing['balance'] > 556) & (df_marketing['balance'] <= 2000),
    (df_marketing['balance'] > 2000)
]
opciones_balance = ['Negative or Zero', 'Low Balance', 'Medium Balance', 'High Balance']

df_marketing['balance_tier'] = np.select(condiciones_balance, opciones_balance, default='Negative or Zero')

# Mapeo de Job a 5 Macro-categorías (aquí separé en un principio a unemployed a una categoría aparte. 
# Pero me dio como resultado que no era distinto a retired y student, entonces van en la misma categoria)
def agrupar_job(job):
    job = str(job).lower().strip()
    if job in ['admin.', 'management','blue-collar' ]:
        return 'Asalariados'
    elif job in ['technician', 'services', 'housemaid']:
        return 'Operativos'
    elif job in ['self-employed', 'entrepreneur']:
        return 'Independientes'
    elif job in ['retired', 'student', 'unemployed']:
        return 'Inactivos'
    else:
        return 'Unknown'
df_marketing['job_group'] = df_marketing['job'].apply(agrupar_job)


In [23]:
# Convertir meses de texto a números
meses = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6, "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}

df_marketing["month_num"] = df_marketing["month"].map(meses)

# Calcular los años basándose en el cambio de diciembre a enero
years = []
current_year = 2008
previous_month = df_marketing["month_num"].iloc[0]

for month in df_marketing["month_num"]:
    if previous_month == 12 and month == 1:
        current_year += 1

    years.append(current_year)
    previous_month = month

df_marketing["year"] = years

# Crear columna de fecha completa
df_marketing["date"] = pd.to_datetime({"year": df_marketing["year"], "month": df_marketing["month_num"], "day": df_marketing["day"]}, errors="coerce")

# Obtener y traducir el día de la semana en orden cronológico
dias_traducidos = {"Monday": "Lunes", "Tuesday": "Martes", "Wednesday": "Miércoles", "Thursday": "Jueves", "Friday": "Viernes", "Saturday": "Sábado", "Sunday": "Domingo"}
orden = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
df_marketing["day_of_week"] = pd.Categorical(df_marketing["date"].dt.day_name().map(dias_traducidos), categories=orden, ordered=True)

# Imprimir el reporte de control de fechas
print(f"Fechas nulas: {df_marketing['date'].isna().sum()}")
print(f"Rango: {df_marketing['date'].min()} a {df_marketing['date'].max()}")


Fechas nulas: 0
Rango: 2008-05-05 00:00:00 a 2010-12-29 00:00:00


In [24]:
# Conteo de total de dias de la semana del dataset
df_marketing['day_of_week'].value_counts()

Jueves       2390
Viernes      2259
Miércoles    2126
Martes       1410
Sábado       1303
Lunes         861
Domingo       813
Name: day_of_week, dtype: int64

In [25]:
df_marketing['day_of_week'].value_counts().sum()

11162

In [ ]:
dtale.show(df_marketing)


In [26]:
# Diagnóstico: composición de job_group por día para saber porque Lunes tiene tan buen comportamiento
composicion = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["job_group"],
    normalize="index"
) * 100

print(composicion.round(1))

job_group    Asalariados  Inactivos  Independientes  Operativos  Unknown
day_of_week                                                             
Lunes               47.7       19.3             6.7        25.6      0.7
Martes              50.9       16.9             6.0        25.7      0.6
Miércoles           52.0       14.9             6.3        26.2      0.7
Jueves              51.1       13.3             6.9        27.9      0.8
Viernes             54.4       12.8             5.4        26.9      0.5
Sábado              55.2        9.2             7.0        28.4      0.2
Domingo             54.4        5.9             9.7        29.3      0.7


In [27]:
# Diagnóstico: composición de debt_profile por día
composicion = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["debt_profile"],
    normalize="index"
) * 100

print(composicion.round(1))

debt_profile  No Debt  With Debt
day_of_week                     
Lunes            61.0       39.0
Martes           51.1       48.9
Miércoles        48.4       51.6
Jueves           47.6       52.4
Viernes          45.9       54.1
Sábado           37.5       62.5
Domingo          39.1       60.9


In [28]:
# Diagnóstico: composición de poutcome por día
composicion = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["poutcome"],
    normalize="index"
) * 100

print(composicion.round(1))

poutcome     failure  no_campaign  other  success
day_of_week                                      
Lunes           12.9         62.5    6.7     17.9
Martes          17.3         60.7    6.8     15.2
Miércoles       10.0         76.1    3.4     10.5
Jueves          11.3         72.9    5.2     10.6
Viernes         10.5         74.9    5.6      8.9
Sábado           9.7         84.7    4.0      1.7
Domingo          3.3         95.4    1.1      0.1


In [29]:
# Diagnóstico: composición de balance_tier por día
composicion = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["balance_tier"],
    normalize="index"
) * 100

print(composicion.round(1))

balance_tier  High Balance  Low Balance  Medium Balance  Negative or Zero
day_of_week                                                              
Lunes                 25.4         33.8            31.6               9.2
Martes                24.3         35.5            29.9              10.4
Miércoles             21.0         36.9            27.2              14.8
Jueves                22.2         36.8            28.8              12.2
Viernes               21.9         38.2            27.0              12.9
Sábado                19.0         38.5            26.6              15.9
Domingo               19.8         40.0            24.0              16.2


In [30]:
# Diagnóstico: composición de contact por día
composicion = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["contact"],
    normalize="index"
) * 100

print(composicion.round(1))

contact      cellular  telephone  unknown
day_of_week                              
Lunes            82.3        8.2      9.4
Martes           86.2        8.7      5.1
Miércoles        71.6        7.1     21.3
Jueves           74.1        6.7     19.2
Viernes          72.1        6.4     21.5
Sábado           65.1        7.8     27.1
Domingo          42.8        2.8     54.4


In [34]:
# Forzar el orden de los días de la semana
dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
df_marketing["day_of_week"] = pd.Categorical(df_marketing["day_of_week"], categories=dias, ordered=True)

# Calcular e imprimir la tabla de días calendario por año
tabla_dias = pd.crosstab(index=df_marketing['year'], columns=df_marketing['day_of_week'])
print("\n días calendario por año \n")
print(tabla_dias)

# Calcular e imprimir la tabla de llamadas totales por año
tabla_llamadas = df_marketing.groupby(['year', 'day_of_week'], observed=False)['campaign'].sum().unstack(fill_value=0)
print("\n \n llamadas totales por año \n")
print(tabla_llamadas)



 días calendario por año 

day_of_week  Lunes  Martes  Miércoles  Jueves  Viernes  Sábado  Domingo
year                                                                   
2008           187     184        248     239      227       1        3
2009           393     373        372     460      368       0        0
2010           281     853       1506    1691     1664    1302      810

 
 llamadas totales por año 

day_of_week  Lunes  Martes  Miércoles  Jueves  Viernes  Sábado  Domingo
year                                                                   
2008           607     492        727     672      641       1        3
2009           802     629        661     824      726       0        0
2010           568    1774       4093    4016     4241    4000     2522


In [35]:
# CHI-CUADRADO DE INDEPENDENCIA

# Crear tabla de cruce (usando la columna mapeada 'deposit')
tabla = pd.crosstab(
    df_marketing["day_of_week"],
    df_marketing["deposit"].map({1: "Convierte", 0: "No convierte"})
)
print("TABLA CONTINGENCIA:\n", tabla)

# Prueba estadística Chi-cuadrado y validación de resultados 
chi2, p, _, esperadas = chi2_contingency(tabla)
print(f"\nCHI2: {chi2:.4f} | P-valor: {p:.4f}")
print("Conclusión:", "Significativo" if p < 0.05 else "No asociado")
print(f"Celdas esperadas < 5: {(esperadas < 5).sum()}")

# Tasa de conversión porcentual por día 
tasa = df_marketing.groupby("day_of_week", observed=True)["deposit"].agg(conv="sum", total="count")
tasa["tasa_%"] = (tasa["conv"] / tasa["total"] * 100).round(2)
print("\nTASA CONVERSIÓN:\n", tasa)

# Residuos estandarizados (para identificar qué días contribuyen más a la asociación)
residuos = pd.DataFrame(
    (tabla.values - esperadas) / np.sqrt(esperadas),
    index=tabla.index,
    columns=tabla.columns
)
print("\nRESIDUOS ESTANDARIZADOS:\n", residuos.round(3))

TABLA CONTINGENCIA:
 deposit      Convierte  No convierte
day_of_week                         
Lunes              812            49
Martes             949           461
Miércoles         1071          1055
Jueves            1205          1185
Viernes           1010          1249
Sábado             164          1139
Domingo             78           735

CHI2: 2106.3355 | P-valor: 0.0000
Conclusión: Significativo
Celdas esperadas < 5: 0

TASA CONVERSIÓN:
              conv  total  tasa_%
day_of_week                     
Lunes         812    861   94.31
Martes        949   1410   67.30
Miércoles    1071   2126   50.38
Jueves       1205   2390   50.42
Viernes      1010   2259   44.71
Sábado        164   1303   12.59
Domingo        78    813    9.59

RESIDUOS ESTANDARIZADOS:
 deposit      Convierte  No convierte
day_of_week                         
Lunes           20.003       -18.982
Martes          10.867       -10.312
Miércoles        2.004        -1.902
Jueves           2.155        -2.

In [45]:
# ---MODELO REGRESIÓN LOGÍSTICA---

# Establecemos 'Domingo', 'inactivos', 'With Debt' y 'Negative or Zero' 
# como las categorías base de comparación neutral para cada variable categórica respectivamente

formula_timing = """
deposit ~ C(day_of_week, Treatment(reference='Domingo')) 
         + C(job_group, Treatment(reference='Inactivos')) 
         + C(debt_profile, Treatment(reference='With Debt')) 
         + C(balance_tier, Treatment(reference='Negative or Zero')) 
         + C(poutcome)
         + C(contact, Treatment(reference="telephone"))
"""

modelo_timing = smf.logit(formula_timing, data=df_marketing).fit()

# Extracción y Estructuración de Resultados (Odds Ratios)
resultados_completos = pd.DataFrame({
    'Odds Ratio (OR)': np.exp(modelo_timing.params),
    'P-valor': modelo_timing.pvalues.round(4)
})

Optimization terminated successfully.
         Current function value: 0.522112
         Iterations 7


In [46]:
# Ver el resumen estadístico completo
modelo_timing = smf.logit(formula_timing, data=df_marketing).fit()
print(modelo_timing.summary())

Optimization terminated successfully.
         Current function value: 0.522112
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                deposit   No. Observations:                11162
Model:                          Logit   Df Residuals:                    11142
Method:                           MLE   Df Model:                           19
Date:                Thu, 11 Jun 2026   Pseudo R-squ.:                  0.2453
Time:                        14:14:50   Log-Likelihood:                -5827.8
converged:                       True   LL-Null:                       -7721.6
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                                 coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------

In [47]:
# Resultados (Odds Ratios)
display(resultados_completos)

,Odds Ratio (OR),P-valor
Intercept,0.093461,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Lunes]",98.475556,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Martes]",11.257261,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Miércoles]",6.581487,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Jueves]",6.420335,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Viernes]",5.193277,0.0000
"C(day_of_week, Treatment(reference='Domingo'))[T.Sábado]",1.059198,0.7003
"C(job_group, Treatment(reference='Inactivos'))[T.Asalariados]",0.661022,0.0000
"C(job_group, Treatment(reference='Inactivos'))[T.Independientes]",0.597647,0.0000
"C(job_group, Treatment(reference='Inactivos'))[T.Operativos]",0.602308,0.0000
